# First Set Up the GPU in Kaggle 

In [1]:
!nvidia-smi # to check if gpu is loaded for current session

Sun Apr 19 13:45:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Download PreRequisits

In [2]:
%%writefile requirements.txt

--extra-index-url https://download.pytorch.org/whl/cu124
torch==2.5.1+cu124
torchvision==0.20.1+cu124
accelerate>=1.1.0
tqdm>=4.66.0
numpy>=1.24.3,<2.0.0

Writing requirements.txt


In [3]:
!pip list | grep -E "torch|torchvision|accelerate|numpy|tqdm"

accelerate                               1.12.0
numpy                                    2.0.2
pytorch-ignite                           0.5.3
pytorch-lightning                        2.6.1
torch                                    2.10.0+cu128
torchao                                  0.10.0
torchaudio                               2.10.0+cu128
torchcodec                               0.10.0+cu128
torchdata                                0.11.0
torchinfo                                1.8.0
torchmetrics                             1.9.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.25.0+cu128
tqdm                                     4.67.3


In [4]:
#!pip install -r requirements.txt

In [5]:
%%writefile dataset-cifar.py

import argparse
import torchvision.datasets as datasets
import torchvision.transforms as transforms

parser = argparse.ArgumentParser()
parser.add_argument("--data-dir", type=str, default="../../datasets/data-cifar")
args = parser.parse_args()

print(f"[*] Downloading CIFAR-10 to {args.data_dir} ...")
datasets.CIFAR10(args.data_dir, train=True,  download=True, transform=transforms.ToTensor())
datasets.CIFAR10(args.data_dir, train=False, download=True, transform=transforms.ToTensor())
print("[✓] CIFAR-10 download complete.")

Writing dataset-cifar.py


In [6]:
!python dataset-cifar.py

[*] Downloading CIFAR-10 to ../../datasets/data-cifar ...
100%|████████████████████████████████████████| 170M/170M [00:03<00:00, 54.0MB/s]
[✓] CIFAR-10 download complete.


In [7]:
!ls ../../datasets/ # verify comman

data-cifar


# Single GPU Setup

In [8]:
%%writefile resnet-cifar.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model
    

def train(model, train_loader, loss_fn, optimizer, epoch, device):
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
            f"| Acc: {100*correct/total_samples:.2f}% "
            f"| Time: {(time.time()-start):.2f}s")


def test(model, test_loader, loss_fn, epoch, device):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)


    print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
            f"| Acc: {100*correct/total_samples:.2f}% "
            f"| Time: {time.time()-start:.2f}s")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",      type=int,   default=10)
    parser.add_argument("--batch-size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--num-workers", type=int,   default=8)
    parser.add_argument("--data-dir",    type=str,   default="../../datasets/data-cifar")  # ← new
    args = parser.parse_args()


    device = torch.device(f"cuda")
    

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    # ← use args.data_dir instead of hardcoded relative path
    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)


    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                               num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=args.batch_size * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model = build_model().to(device)
   
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train( model, train_loader, loss_fn, optimizer, epoch, device)
        test( model, test_loader, loss_fn, epoch, device)
        scheduler.step()

    total = time.time() - total_start
    print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")

if __name__ == "__main__":
    main()

Writing resnet-cifar.py


In [9]:
!python resnet-cifar.py

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch [1] | Train Loss: 2.6982 | Acc: 12.4

## Multi GPU DDP Setup

In [13]:
%%writefile cifar-resnet-ddp.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train(rank, model, train_loader, train_sampler, loss_fn, optimizer, epoch, device):
    train_sampler.set_epoch(epoch)
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    if rank == 0:
        print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def test(rank, model, test_loader, loss_fn, epoch, device):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)

    if rank == 0:
        print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",      type=int,   default=10)
    parser.add_argument("--batch-size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--num-workers", type=int,   default=8)
    parser.add_argument("--data-dir",    type=str,   default="../../datasets/data-cifar")  # ← new
    args = parser.parse_args()

    dist.init_process_group(backend="nccl")
    rank       = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(local_rank)

    if rank == 0:
        print(f"[INFO] World size: {world_size} | Effective batch: {args.batch_size * world_size}")
        print(f"[INFO] Data dir:   {args.data_dir}")

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    # ← use args.data_dir instead of hardcoded relative path
    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)

    train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                              sampler=train_sampler, num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=args.batch_size * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model = build_model().to(device)
    model = DDP(model, device_ids=[local_rank])

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train(rank, model, train_loader, train_sampler, loss_fn, optimizer, epoch, device)
        test(rank, model, test_loader, loss_fn, epoch, device)
        scheduler.step()

    if rank == 0:
        total = time.time() - total_start
        print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing cifar-resnet-ddp.py


In [14]:
! torchrun \
    --nnodes=1 \
    --nproc-per-node=2 \
    cifar-resnet-ddp.py \
    --epochs=10 \
    --batch-size=512

W0418 15:14:21.579000 507 torch/distributed/run.py:852] 
W0418 15:14:21.579000 507 torch/distributed/run.py:852] *****************************************
W0418 15:14:21.579000 507 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0418 15:14:21.579000 507 torch/distributed/run.py:852] *****************************************
[INFO] World size: 2 | Effective batch: 1024
[INFO] Data dir:   ../../datasets/data-cifar
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to

# Multi GPU DDP Accelerate Setup

In [15]:
%%writefile cifar-resnet-ddp-accelerate.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from accelerate import Accelerator
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train(accelerator, model, train_loader, loss_fn, optimizer, epoch):
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        accelerator.backward(loss)
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    if accelerator.is_main_process:
        print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def test(accelerator, model, test_loader, loss_fn, epoch):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)

    if accelerator.is_main_process:
        print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",      type=int,   default=10)
    parser.add_argument("--batch-size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--num-workers", type=int,   default=8)
    parser.add_argument("--data-dir",    type=str,   default="../../datasets/data-cifar")
    args = parser.parse_args()

    torch.manual_seed(42)

    # Create an Accelerator object — handles device placement and distributed setup
    accelerator = Accelerator()

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                              shuffle=True, num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=args.batch_size * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model     = build_model()
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    # Accelerator wraps model, optimizer, and dataloaders — DDP is handled automatically
    model, optimizer, train_loader, test_loader = accelerator.prepare(
        model, optimizer, train_loader, test_loader
    )

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train(accelerator, model, train_loader, loss_fn, optimizer, epoch)
        test(accelerator, model, test_loader, loss_fn, epoch)
        scheduler.step()

    if accelerator.is_main_process:
        total = time.time() - total_start
        print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")


if __name__ == "__main__":
    main()

Writing cifar-resnet-ddp-accelerate.py


In [16]:
! accelerate launch \
    --num_processes 2 \
    cifar-resnet-ddp-accelerate.py \
    --epochs 10 \
    --batch-size 512

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/datal

# Multi GPU FSDP Setup

In [17]:
%%writefile cifar-resnet-fsdp.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# FSDP packages
from torch.utils.data import DistributedSampler
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
    BackwardPrefetch,
    ShardingStrategy,
    CPUOffload,
)
from torch.distributed.fsdp.wrap import (
    size_based_auto_wrap_policy,
    enable_wrap,
    wrap,
)
import functools
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train(rank, model, train_loader, train_sampler, loss_fn, optimizer, epoch, device):
    train_sampler.set_epoch(epoch)
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    if rank == 0:
        print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def test(rank, model, test_loader, loss_fn, epoch, device):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)

    if rank == 0:
        print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def setup():
    dist.init_process_group(backend="nccl")


def cleanup():
    dist.destroy_process_group()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",           type=int,   default=10)
    parser.add_argument("--batch-size",       type=int,   default=512)
    parser.add_argument("--lr",               type=float, default=0.1)
    parser.add_argument("--num-workers",      type=int,   default=8)
    parser.add_argument("--data-dir",         type=str,   default="../../datasets/data-cifar")
    parser.add_argument("--sharding-strategy",type=str,   default="FULL_SHARD",
                        choices=["FULL_SHARD", "SHARD_GRAD_OP", "NO_SHARD"],
                        help="FSDP sharding strategy")
    parser.add_argument("--mixed-precision",  action="store_true", help="Use mixed precision (fp16)")
    parser.add_argument("--cpu-offload",      action="store_true", help="Offload parameters to CPU")
    args = parser.parse_args()

    torch.manual_seed(42)

    setup()

    rank       = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(local_rank)

    if rank == 0:
        print(f"[INFO] Using {world_size} GPUs")
        print(f"[INFO] Sharding Strategy: {args.sharding_strategy}")
        print(f"[INFO] Mixed Precision:   {args.mixed_precision}")
        print(f"[INFO] CPU Offload:       {args.cpu_offload}")
        print(f"[INFO] Data dir:          {args.data_dir}")

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)

    train_sampler = DistributedSampler(train_dataset, num_replicas=world_size, rank=rank, shuffle=True)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                              sampler=train_sampler, num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=args.batch_size * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model = build_model().to(device)

    # Auto-wrap policy: shard any sub-module with >= 1M parameters
    my_auto_wrap_policy = functools.partial(
        size_based_auto_wrap_policy, min_num_params=1_000_000
    )

    # Sharding strategy
    sharding_map = {
        "FULL_SHARD":    ShardingStrategy.FULL_SHARD,
        "SHARD_GRAD_OP": ShardingStrategy.SHARD_GRAD_OP,
        "NO_SHARD":      ShardingStrategy.NO_SHARD,
    }
    sharding_strategy = sharding_map[args.sharding_strategy]

    # Mixed precision config
    mixed_precision_config = None
    if args.mixed_precision:
        mixed_precision_config = MixedPrecision(
            param_dtype=torch.float16,
            reduce_dtype=torch.float16,
            buffer_dtype=torch.float16,
        )

    # CPU offload config
    cpu_offload = CPUOffload(offload_params=True) if args.cpu_offload else None

    # Wrap model with FSDP
    model = FSDP(
        model,
        auto_wrap_policy=my_auto_wrap_policy,
        sharding_strategy=sharding_strategy,
        mixed_precision=mixed_precision_config,
        cpu_offload=cpu_offload,
        device_id=local_rank,
        backward_prefetch=BackwardPrefetch.BACKWARD_PRE,
    )

    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train(rank, model, train_loader, train_sampler, loss_fn, optimizer, epoch, device)
        test(rank, model, test_loader, loss_fn, epoch, device)
        scheduler.step()

    if rank == 0:
        total = time.time() - total_start
        print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")

    cleanup()


if __name__ == "__main__":
    main()

Writing cifar-resnet-fsdp.py


In [18]:
! accelerate launch \
    --num_processes 2 \
    cifar-resnet-fsdp.py \
      --epochs=10 \
      --batch-size=512 \
      --lr=0.1 \
      --num-workers=8 \
      --sharding-strategy=FULL_SHARD

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[INFO] Using 2 GPUs
[INFO] Sharding Strategy: FULL_SHARD
[INFO] Mixed Precision:   False
[INFO] CPU Offload:       False
[INFO] Data dir:          ../../datasets/data-cifar
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, low

# Multi GPU FSDP Accelerate Setup

In [10]:
%%writefile cifar-resnet-fsdp-accelerate.py

import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from accelerate import Accelerator
import argparse


def build_model():
    model = models.resnet34(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train(accelerator, model, train_loader, loss_fn, optimizer, epoch):
    model.train()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    for x, y in train_loader:
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred, y)
        accelerator.backward(loss)
        optimizer.step()

        total_loss += loss.item()
        correct += (pred.argmax(1) == y).sum().item()
        total_samples += y.size(0)

    if accelerator.is_main_process:
        print(f"Epoch [{epoch}] | Train Loss: {total_loss/len(train_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def test(accelerator, model, test_loader, loss_fn, epoch):
    model.eval()
    total_loss, correct, total_samples = 0, 0, 0
    start = time.time()

    with torch.no_grad():
        for x, y in test_loader:
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()
            total_samples += y.size(0)

    if accelerator.is_main_process:
        print(f"Epoch [{epoch}] | Test  Loss: {total_loss/len(test_loader):.4f} "
              f"| Acc: {100*correct/total_samples:.2f}% "
              f"| Time: {time.time()-start:.2f}s")


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs",      type=int,   default=10)
    parser.add_argument("--batch-size",  type=int,   default=512)
    parser.add_argument("--lr",          type=float, default=0.1)
    parser.add_argument("--num-workers", type=int,   default=8)
    parser.add_argument("--data-dir",    type=str,   default="../../datasets/data-cifar")
    args = parser.parse_args()

    torch.manual_seed(42)

    # Accelerator reads the FSDP config from custom_config.yaml (passed via --config_file in the launch command)
    accelerator = Accelerator()

    if accelerator.is_main_process:
        print(f"[INFO] Distributed type: {accelerator.distributed_type}")
        print(f"[INFO] Num processes:    {accelerator.num_processes}")
        print(f"[INFO] Data dir:         {args.data_dir}")

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])

    train_dataset = datasets.CIFAR10(args.data_dir, train=True,  download=False, transform=train_tf)
    test_dataset  = datasets.CIFAR10(args.data_dir, train=False, download=False, transform=test_tf)

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                              shuffle=True, num_workers=args.num_workers,
                              pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_dataset,  batch_size=args.batch_size * 2,
                              shuffle=False, num_workers=args.num_workers,
                              pin_memory=True)

    model     = build_model()
    loss_fn   = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)

    # Accelerator handles FSDP sharding as defined in custom_config.yaml
    model, optimizer, train_loader, test_loader = accelerator.prepare(
        model, optimizer, train_loader, test_loader
    )

    total_start = time.time()
    for epoch in range(1, args.epochs + 1):
        train(accelerator, model, train_loader, loss_fn, optimizer, epoch)
        test(accelerator, model, test_loader, loss_fn, epoch)
        scheduler.step()

    if accelerator.is_main_process:
        total = time.time() - total_start
        print(f"\nTotal time: {total:.2f}s ({total/60:.2f} min)")


if __name__ == "__main__":
    main()

Writing cifar-resnet-fsdp-accelerate.py


In [11]:
! accelerate launch \
    --num_processes 2 \
    cifar-resnet-fsdp-accelerate.py \
    --epochs 10 \
    --batch-size 512

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
[INFO] Distributed type: DistributedType.MULTI_GPU
[INFO] Num processes:    2
[INFO] Data dir:         ../../datasets/data-cifar
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slow

In [ ]:
--extra-index-url https://download.pytorch.org/whl/cu117
torch
torchvision
accelerate>=1.1.0
tqdm>=4.66.0
numpy>=1.24.3,<2.0.0